# Inference Pipeline — End-to-End Demo

**Sovereign Dialect-Bridge** — demo lengkap pipeline NLP yang dipakai backend Django.

Notebook ini menjalankan pipeline yang sama dengan production untuk validasi & demo:

```
INPUT TEKS (dialek)
    ↓
[Stage 1] detect_dialect       → kode dialek (jv, su, min, ...) + confidence
    ↓
[Stage 2] translate_to_indonesian → teks Bahasa Indonesia
    ↓
[Stage 3] summarize            → ringkasan (mT5-base) atau fallback TextRank
    ↓
[Stage 4] extract_entities     → LOC / PER / ORG dari cahya/bert-base-indonesian-NER
    ↓
[Stage 5] classify_category    → 8 kategori (Infrastruktur, Kesehatan, dll)
    ↓
[Stage 6] score_urgency        → low / medium / high / critical (weighted keywords)
    ↓
[Stage 7] extract_keywords     → top 5 kata kunci
    ↓
OUTPUT: NLPResult lengkap (dialect, summary, entities, urgency, category, keywords, confidence)
```

Semua model di-load **satu kali** di awal. Selanjutnya inference per teks ~5-15 detik di CPU.

## 1. Setup Device & Imports

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import time
from pathlib import Path
from dataclasses import dataclass, field

import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline as hf_pipeline

# Device: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    print("  [INFO] CPU — inference per teks ~10-30 detik")

/opt/homebrew/Caskroom/miniforge/base/envs/ai_core/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0607 00:55:38.008000 24583 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Device: mps


In [2]:
NOTEBOOK_DIR = Path.cwd()
EXPERIMENT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebook" else NOTEBOOK_DIR
PROJECT_ROOT = EXPERIMENT_ROOT.parent

DIALECT_MODEL_PATH = EXPERIMENT_ROOT / "dialect_detector" / "dialect_detector.joblib"
MT5_MODEL_DIR = EXPERIMENT_ROOT / "model" / "mt5base"
if not MT5_MODEL_DIR.exists():
    # fallback ke backend
    MT5_MODEL_DIR = PROJECT_ROOT / "backend" / "models" / "mt5base"

print(f"Dialect detector : {DIALECT_MODEL_PATH}  ({'OK' if DIALECT_MODEL_PATH.exists() else 'MISSING'})")
print(f"mT5-base path    : {MT5_MODEL_DIR}  ({'OK' if MT5_MODEL_DIR.exists() else 'MISSING'})")

Dialect detector : /Users/fadh/Documents/BINUS/Semester 4/Sovereign-Dialect-Bridge/experiment/dialect_detector/dialect_detector.joblib  (OK)
mT5-base path    : /Users/fadh/Documents/BINUS/Semester 4/Sovereign-Dialect-Bridge/experiment/model/mt5base  (OK)


## 2. Stage 1 — Dialect Detection

Load TF-IDF + LogReg classifier yang sudah di-train di notebook `SE complaint classifier.ipynb`.

In [3]:
DIALECT_NAMES = {
    "id":  "Bahasa Indonesia", "jv":  "Bahasa Jawa",
    "su":  "Bahasa Sunda",     "min": "Minangkabau",
    "ace": "Aceh",             "ban": "Bali",
    "bjn": "Banjar",           "bug": "Bugis",
    "mad": "Madura",           "nij": "Ngaju Dayak",
    "bbc": "Batak Toba",       "en":  "English",
    "xx":  "Tidak Terdeteksi",
}

MIN_DIALECT_CONF = 0.35    # di bawah ini → return 'xx'

_dialect_clf = joblib.load(DIALECT_MODEL_PATH)
print(f"Dialect detector loaded — {len(_dialect_clf.classes_)} kelas: {list(_dialect_clf.classes_)}")

Dialect detector loaded — 12 kelas: ['ace', 'ban', 'bbc', 'bjn', 'bug', 'en', 'id', 'jv', 'mad', 'min', 'nij', 'su']


In [4]:
def detect_dialect(text: str) -> tuple[str, float]:
    """Return (kode_dialek, confidence). 'xx' kalau teks < 10 char atau low confidence."""
    cleaned = text.strip()
    if len(cleaned) < 10:
        return "xx", 0.0
    probs = _dialect_clf.predict_proba([cleaned])[0]
    classes = list(_dialect_clf.classes_)
    top_idx = int(probs.argmax())
    top_label, top_conf = classes[top_idx], float(probs[top_idx])
    if top_conf >= MIN_DIALECT_CONF:
        return top_label, round(top_conf, 3)
    return "xx", round(top_conf, 3)

# Quick test
for txt in [
    "Dalane ning ngarep omah rusak parah",
    "Kunaon di Kopo teh sok macet wae jalanna",
    "Pemerintah harus memperbaiki jalan ini segera",
    "Halo",   # terlalu pendek
]:
    code, conf = detect_dialect(txt)
    print(f"  [{code:3s}] conf={conf:.3f}  |  {DIALECT_NAMES[code]:18s}  |  '{txt[:50]}'")

  [jv ] conf=0.730  |  Bahasa Jawa         |  'Dalane ning ngarep omah rusak parah'
  [su ] conf=0.495  |  Bahasa Sunda        |  'Kunaon di Kopo teh sok macet wae jalanna'
  [id ] conf=0.630  |  Bahasa Indonesia    |  'Pemerintah harus memperbaiki jalan ini segera'
  [xx ] conf=0.000  |  Tidak Terdeteksi    |  'Halo'


## 3. Stage 2 — Translate ke Bahasa Indonesia

Pakai `deep_translator.GoogleTranslator` yang ringan & tanpa key. Backend juga punya fallback ke `googletrans` kalau gagal.

In [5]:
# Mapping kode internal -> source language code untuk Google Translate
DIALECT_TO_GOOGLE = {
    "jv": "jw", "su": "su", "min": "auto",
    "ace": "auto", "ban": "auto", "bjn": "auto",
    "bug": "auto", "mad": "auto", "nij": "auto",
    "bbc": "auto", "en": "en",
}

def translate_to_indonesian(text: str, dialect: str) -> str:
    """Translate ke BI kalau dialect bukan 'id'/'xx'. Fallback graceful kalau API gagal."""
    if dialect in ("id", "xx"):
        return text
    src = DIALECT_TO_GOOGLE.get(dialect, "auto")
    try:
        from deep_translator import GoogleTranslator
        out = GoogleTranslator(source=src, target="id").translate(text[:4500])
        if out and out.strip():
            return out
    except Exception as e:
        print(f"  [warn] deep_translator failed: {e}")
    return text   # fallback: return as-is

# Quick test
for txt, code in [
    ("Dalane ning ngarep omah rusak parah wis suwe ora dibenahi", "jv"),
    ("Kunaon di Kopo teh sok macet wae jalanna", "su"),
    ("Ambo malaporan jalan di muko rumah ambo rusak bana", "min"),
]:
    print(f"  [{code}] {txt[:60]}")
    print(f"  [id] {translate_to_indonesian(txt, code)[:80]}")
    print()

  [jv] Dalane ning ngarep omah rusak parah wis suwe ora dibenahi
  [warn] deep_translator failed: No module named 'deep_translator'
  [id] Dalane ning ngarep omah rusak parah wis suwe ora dibenahi

  [su] Kunaon di Kopo teh sok macet wae jalanna
  [warn] deep_translator failed: No module named 'deep_translator'
  [id] Kunaon di Kopo teh sok macet wae jalanna

  [min] Ambo malaporan jalan di muko rumah ambo rusak bana
  [warn] deep_translator failed: No module named 'deep_translator'
  [id] Ambo malaporan jalan di muko rumah ambo rusak bana



## 4. Stage 3 — Summarize dengan mT5-base

Model yang sudah ada di `experiment/model/mt5base/` (atau `backend/models/mt5base/`). Strategy:
- **Teks pendek (< 40 kata)** → ambil 2 kalimat pertama (TextRank tidak banyak gunanya)
- **Teks panjang** → mT5 abstractive summarization
- **mT5 gagal** → fallback TextRank (sklearn TF-IDF + networkx PageRank)

In [6]:
MIN_WORDS_NEURAL = 40

print("Loading mT5-base... (~30 detik first time)")
t0 = time.time()
_summarizer_tok = AutoTokenizer.from_pretrained(str(MT5_MODEL_DIR))
_summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(str(MT5_MODEL_DIR)).to(DEVICE).eval()
print(f"  mT5 loaded ({time.time()-t0:.1f}s)")

Loading mT5-base... (~30 detik first time)


KeyboardInterrupt: 

In [ ]:
def _postprocess_mt5(text: str) -> str:
    """Bersihkan artifact mT5 (CamelCase merge, capitalize first letter)."""
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    if text:
        text = text[0].upper() + text[1:]
    text = re.sub(r"([.!?]\s)([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)
    return text


def _summarize_textrank(text: str, n_sents: int = 3) -> str:
    """Fallback extractive summarization."""
    try:
        import numpy as np
        import networkx as nx
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity

        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
        if len(sents) <= n_sents:
            return " ".join(sents) or text.strip()
        vec = TfidfVectorizer().fit_transform(sents)
        sim = cosine_similarity(vec, vec); np.fill_diagonal(sim, 0)
        scores = nx.pagerank(nx.from_numpy_array(sim))
        ranked = sorted(scores, key=scores.get, reverse=True)[:n_sents]
        return " ".join(sents[i] for i in sorted(ranked))
    except Exception:
        sents = re.split(r"(?<=[.!?])\s+", text.strip())
        return " ".join(sents[:2]).strip() or text.strip()


def summarize(text: str) -> str:
    """mT5 untuk teks panjang, TextRank fallback."""
    n_words = len(text.split())
    if n_words < MIN_WORDS_NEURAL:
        sents = re.split(r"(?<=[.!?])\s+", text.strip())
        return " ".join(sents[:2]).strip() or text.strip()

    try:
        prompt = "ringkas: " + text[:1024]
        enc = _summarizer_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            out = _summarizer_model.generate(
                **enc, max_new_tokens=150, num_beams=2,
                no_repeat_ngram_size=3, early_stopping=True, length_penalty=1.0,
            )
        return _postprocess_mt5(_summarizer_tok.decode(out[0], skip_special_tokens=True))
    except Exception as e:
        print(f"  [warn] mT5 failed: {e}")
        return _summarize_textrank(text)

# Test
long_text = (
    "Pemerintah daerah harus segera memperbaiki jalan yang rusak di kawasan ini. "
    "Banyak kendaraan yang mengalami kerusakan akibat melewati jalan berlubang. "
    "Warga sudah mengeluhkan kondisi ini sejak lama tetapi belum ada tindakan. "
    "Selain itu drainase juga buruk sehingga banjir saat hujan deras. "
    "Kami berharap dinas pekerjaan umum segera turun tangan."
)
print("INPUT:", long_text[:100], "...")
print("\nSUMMARY:", summarize(long_text))

## 5. Stage 4 — Named Entity Recognition

Model `cahya/bert-base-indonesian-NER` dari HuggingFace mendeteksi entity:
- **PER** — Orang
- **LOC** — Lokasi
- **ORG** — Organisasi/Instansi
- **NOR**, **CRD**, **MISC** — lainnya

Kita filter score > 0.6 + dedup.

In [ ]:
print("Loading NER... (download model pertama kali ~5 detik)")
t0 = time.time()
_ner_pipeline = hf_pipeline(
    "ner",
    model="cahya/bert-base-indonesian-NER",
    aggregation_strategy="simple",
    device=0 if DEVICE == "cuda" else -1,
)
print(f"  NER loaded ({time.time()-t0:.1f}s)")

In [ ]:
def extract_entities(text: str) -> list:
    """Return list of {text, label, score}."""
    try:
        raw = _ner_pipeline(text[:512])
        seen = set()
        out = []
        for e in raw:
            word = str(e.get("word", "")).strip()
            label = str(e.get("entity_group", "MISC"))
            score = float(e.get("score", 0.0))
            key = (word.lower(), label)
            if score > 0.6 and len(word) > 1 and key not in seen:
                seen.add(key)
                out.append({"text": word, "label": label, "score": round(score, 3)})
        return out
    except Exception as e:
        print(f"  [warn] NER failed: {e}")
        return []

# Test
test_text = "Pak Joko melapor ke Dinas PU Jakarta soal Jalan Sudirman yang berlubang dan banjir."
for ent in extract_entities(test_text):
    print(f"  [{ent['label']}] {ent['text']!r} (score={ent['score']})")

## 6. Stage 5 & 6 — Category & Urgency Scoring (Rule-based)

Tidak butuh model neural — keyword matching cukup akurat untuk 8 kategori dan 4 level urgensi yang kita pakai.

In [ ]:
CATEGORY_KEYWORDS = {
    "Infrastruktur": ["jalan", "jembatan", "drainase", "trotoar", "rusak",
                       "berlubang", "bocor", "ambruk", "gorong", "aspal", "beton"],
    "Kesehatan":     ["puskesmas", "rsud", "rumah sakit", "dokter", "obat",
                       "sanitasi", "air bersih", "stunting", "posyandu"],
    "Pendidikan":    ["sekolah", "guru", "siswa", "buku", "beasiswa",
                       "universitas", "ruang kelas", "pendidikan"],
    "Keamanan":      ["kriminal", "pencurian", "maling", "tawuran", "narkoba",
                       "kekerasan", "perampokan", "begal"],
    "Lingkungan":    ["sampah", "banjir", "polusi", "limbah", "sungai",
                       "bau", "kotor", "pencemaran", "longsor"],
    "Sosial":        ["bantuan", "kemiskinan", "pengangguran", "miskin",
                       "lansia", "disabilitas", "yatim", "pkh"],
    "Administrasi":  ["ktp", "kk", "akta", "sertifikat", "izin",
                       "pungli", "pelayanan", "birokrasi"],
}

URGENCY_KEYWORDS = {
    100: ["kebakaran", "banjir besar", "longsor", "darurat", "nyawa",
          "kematian", "meninggal", "ledakan", "tenggelam", "evakuasi"],
    60:  ["rusak parah", "sudah bertahun", "berbulan-bulan", "ambruk",
          "mendesak", "segera", "tolong cepat"],
    30:  ["rusak", "bocor", "kotor", "tidak berfungsi",
          "masalah", "keluhan", "mati"],
    10:  ["saran", "usul", "semoga", "harap", "minta tolong"],
}


def classify_category(text: str) -> str:
    text_lower = text.lower()
    for cat, kws in CATEGORY_KEYWORDS.items():
        if any(kw in text_lower for kw in kws):
            return cat
    return "Umum"


def score_urgency(text: str) -> str:
    text_lower = text.lower()
    total = sum(score for score, kws in URGENCY_KEYWORDS.items()
                for kw in kws if kw in text_lower)
    if total >= 100: return "critical"
    if total >= 60:  return "high"
    if total >= 30:  return "medium"
    return "low"

# Test
for t in [
    "Jalan rusak parah dan berlubang besar",
    "Kebakaran besar darurat di pasar",
    "Saran perbaikan layanan kelurahan",
    "Banjir besar masuk rumah warga",
]:
    print(f"  category={classify_category(t):14s} urgency={score_urgency(t):8s}  |  '{t}'")

## 7. Stage 7 — Keyword Extraction

Sederhana: tokenize → buang stopwords Bahasa Indonesia → urutkan by (frequency DESC, length DESC) → ambil top N.

In [ ]:
STOPWORDS_ID = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "pada", "adalah",
    "ini", "itu", "tidak", "ada", "juga", "bisa", "akan", "sudah", "telah",
    "atau", "oleh", "dalam", "karena", "saat", "jika", "saya", "kami",
    "kita", "mereka", "dia", "anda", "pun", "saja", "para", "namun",
    "hanya", "lebih", "masih", "belum", "sangat", "sekali", "serta",
}

def extract_keywords(text: str, top_n: int = 5) -> list:
    words = re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())
    freq = {}
    for w in words:
        if w not in STOPWORDS_ID:
            freq[w] = freq.get(w, 0) + 1
    return sorted(freq.keys(), key=lambda w: (-freq[w], -len(w)))[:top_n]

test_text = "Pemerintah harus segera memperbaiki jalan rusak parah di desa ini sebelum banjir tiba"
print(extract_keywords(test_text))

## 8. Full Pipeline Orchestrator

Gabungkan semua stage. Versi notebook ini juga mencatat **timing per stage** untuk profiling.

In [ ]:
@dataclass
class NLPResult:
    dialect: str = "xx"
    dialect_confidence: float = 0.0
    translated_text: str = ""
    summary: str = ""
    named_entities: list = field(default_factory=list)
    keywords: list = field(default_factory=list)
    urgency_level: str = "medium"
    category_name: str = "Umum"
    confidence: float = 0.0
    timing_ms: dict = field(default_factory=dict)


def run_pipeline(raw_text: str, verbose: bool = True) -> NLPResult:
    timing = {}
    def _t(name, fn):
        t0 = time.time()
        result = fn()
        timing[name] = round((time.time() - t0) * 1000)
        if verbose:
            print(f"  [{timing[name]:>5}ms] {name}")
        return result

    if verbose:
        print(f"\n{'='*70}\nPipeline run\n{'='*70}")
        print(f"INPUT ({len(raw_text.split())} kata): {raw_text[:80]}{'...' if len(raw_text)>80 else ''}\n")

    dialect, conf = _t("detect_dialect",      lambda: detect_dialect(raw_text))
    translated    = _t("translate_to_id",     lambda: translate_to_indonesian(raw_text, dialect))
    summary       = _t("summarize",           lambda: summarize(translated))
    entities      = _t("extract_entities",    lambda: extract_entities(translated))
    category      = _t("classify_category",   lambda: classify_category(translated))
    urgency       = _t("score_urgency",       lambda: score_urgency(translated))
    keywords      = _t("extract_keywords",    lambda: extract_keywords(translated))

    final_conf = round(0.4 + 0.15*conf + 0.2 + 0.15, 3)   # baseline + dialect + mT5 + NER

    if verbose:
        total = sum(timing.values())
        print(f"\n  Total: {total} ms ({total/1000:.1f}s)")

    return NLPResult(
        dialect=dialect, dialect_confidence=conf,
        translated_text=translated, summary=summary,
        named_entities=entities, keywords=keywords,
        urgency_level=urgency, category_name=category,
        confidence=final_conf, timing_ms=timing,
    )

## 9. Demo dengan Multiple Inputs

In [ ]:
def show_result(r: NLPResult):
    print(f"\n{'─'*70}")
    print(f"  DIALEK    : {r.dialect}  ({DIALECT_NAMES[r.dialect]})  conf={r.dialect_confidence:.2f}")
    print(f"  KATEGORI  : {r.category_name}")
    print(f"  URGENSI   : {r.urgency_level}")
    print(f"  TRANSLASI : {r.translated_text[:150]}{'...' if len(r.translated_text)>150 else ''}")
    print(f"  RINGKASAN : {r.summary[:200]}")
    if r.named_entities:
        ents = ", ".join(f"{e['text']!r}({e['label']})" for e in r.named_entities[:5])
        print(f"  ENTITAS   : {ents}")
    else:
        print(f"  ENTITAS   : (none)")
    print(f"  KEYWORDS  : {r.keywords}")
    print(f"  CONFIDENCE: {r.confidence}")
    print(f"  TIMING    : {r.timing_ms}")

In [ ]:
# Demo 1: Teks Bahasa Jawa
TEXT_JV = (
    "Dalane ning ngarep omah rusak parah wis suwe banget ora dibenahi karo pemerintah. "
    "Jalan iki dadi gedhe lan jero. Mlaku-mlaku susah, motor rusak. "
    "Tolong segera ditangani supaya warga ora kesusahan terus."
)
show_result(run_pipeline(TEXT_JV))

In [ ]:
# Demo 2: Teks Bahasa Sunda
TEXT_SU = (
    "Kunaon di Kopo teh sok macet wae jalanna utamana mun jam-jam rame. "
    "Mana jalanna sempit, terus di sisi jalanna aya warung-warung, jadi teu bisa dilewatan dua mobil pas papasan. "
    "Urang mah ngarepkeun bisa diatur ulang jalanna, misalna dijadikeun jalan searah atawa dilebarkeun."
)
show_result(run_pipeline(TEXT_SU))

In [ ]:
# Demo 3: Teks Bahasa Minang
TEXT_MIN = (
    "Ambo malaporan jalan di muko rumah ambo rusak bana, alah lamo indak dibeton. "
    "Urang nan lewat di jalan ko sering kacilakaan motonyo dek jalan nan balubuang gadang. "
    "Ambo harok supayo pemerintah daerah capek mambuek jalan tu elok baliak."
)
show_result(run_pipeline(TEXT_MIN))

In [ ]:
# Demo 4: Teks Bahasa Indonesia (skip translation stage)
TEXT_ID = (
    "Pak Joko melapor ke Dinas PU Jakarta bahwa Jalan Sudirman berlubang besar. "
    "Banyak kendaraan rusak akibat lubang ini. Mohon segera diperbaiki sebelum hujan deras tiba. "
    "Warga sangat berharap ada tindakan cepat dari pemerintah daerah."
)
show_result(run_pipeline(TEXT_ID))

In [ ]:
# Demo 5: Teks kritis (urgency=critical)
TEXT_CRITICAL = (
    "Darurat kebakaran besar di pasar tradisional desa kami. Banyak korban warga yang terjebak. "
    "Sudah ada evakuasi tetapi belum semua diselamatkan. Mohon bantuan secepatnya dari BPBD."
)
show_result(run_pipeline(TEXT_CRITICAL))

## 10. Try Your Own Text

Ganti `MY_TEXT` di bawah dengan teks aduan apapun (boleh dialek atau BI baku):

In [ ]:
MY_TEXT = """
Ganti teks ini dengan aduan kamu. Bisa pakai bahasa daerah atau Bahasa Indonesia.
Pipeline akan otomatis mendeteksi dialek, menerjemahkan ke BI, meringkas, dan mengekstrak entitas.
"""

if MY_TEXT.strip() and "Ganti teks" not in MY_TEXT:
    show_result(run_pipeline(MY_TEXT.strip()))
else:
    print("⚠️  Ganti MY_TEXT di atas dengan teks aduan sebelum di-run.")

## 11. Benchmark Total Latency

Berapa lama satu inference (semua 7 stage)?

In [ ]:
import statistics

test_inputs = [TEXT_JV, TEXT_SU, TEXT_MIN, TEXT_ID, TEXT_CRITICAL]
all_timings = []
for txt in test_inputs:
    r = run_pipeline(txt, verbose=False)
    all_timings.append(sum(r.timing_ms.values()))

print(f"Total latency per input (ms): {all_timings}")
print(f"Mean: {statistics.mean(all_timings):.0f} ms")
print(f"Min : {min(all_timings)} ms")
print(f"Max : {max(all_timings)} ms")
print()
print("Breakdown per stage (rata-rata, ms):")
stage_means = {}
for txt in test_inputs:
    r = run_pipeline(txt, verbose=False)
    for stage, t in r.timing_ms.items():
        stage_means.setdefault(stage, []).append(t)
for stage, ts in stage_means.items():
    print(f"  {stage:25s} : mean={statistics.mean(ts):>6.0f}  min={min(ts):>5}  max={max(ts):>5}")

## Kesimpulan Pipeline

| Stage | Tool / Model | Latency CPU | Catatan |
|-------|--------------|-------------|---------|
| 1. Detect dialect | TF-IDF + LogReg | < 5 ms | Local model, 5 MB |
| 2. Translate | deep_translator (Google) | 500-2000 ms | Network call |
| 3. Summarize | mT5-base | 3000-10000 ms | Bottleneck di CPU; cepat di GPU/MPS |
| 4. Extract entities | cahya/bert-NER | 200-500 ms | First-call download dari HF Hub |
| 5. Classify category | Keyword matching | < 1 ms | Rule-based |
| 6. Score urgency | Weighted keyword | < 1 ms | Rule-based |
| 7. Extract keywords | Regex + frequency | < 1 ms | Rule-based |

**Total**: 5-15 detik per teks di CPU, 1-3 detik di MPS/GPU.

Karena ini berjalan di **background thread** Django (lihat `backend/complaints/views.py::_run_nlp_pipeline`), user submit aduan langsung dapat response 201 (< 2 detik). Hasil NLP muncul beberapa detik kemudian via polling frontend.